---
## Chapter 2: Working with Text Data

**Source of truth:** Raschka, *Build a Large Language Model From Scratch*, Ch. 02  
**Mac M4 adjustments:** cells marked ⚡ deviate from the original to use MPS device placement, appropriate dtypes, and macOS-safe DataLoader settings.

This chapter transforms raw text into tensor inputs an LLM can consume:

| Section | Topic |
|---------|-------|
| 2.1 | Word embeddings — the concept |
| 2.2 | Tokenizing text with regex |
| 2.3 | Converting tokens → integer IDs |
| 2.4 | Special tokens (`<\|endoftext\|>`, `<\|unk\|>`) |
| 2.5 | Byte-Pair Encoding (BPE) via tiktoken |
| 2.6 | Sliding-window data sampling + DataLoader |
| 2.7 | Token embedding layers |
| 2.8 | Positional embeddings |


In [1]:
# ⚡ M4 Mac: Device Setup
# Raschka's original code runs on CPU. Here we select MPS (Metal Performance Shaders)
# when available — Apple's GPU compute backend for PyTorch on Apple Silicon.
# In Ch. 02 the payoff is modest (data prep, not training), but establishing
# the device pattern now carries forward into later chapters.

import torch

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"PyTorch version : {torch.__version__}")
print(f"Active device   : {device}")
print(f"MPS available   : {torch.backends.mps.is_available()}")
print(f"MPS built-in    : {torch.backends.mps.is_built()}")

PyTorch version : 2.12.1
Active device   : mps
MPS available   : True
MPS built-in    : True


---
## 2.1 Understanding Word Embeddings

LLMs can't process raw text — they operate on continuous vector representations called **embeddings**.
Each token maps to a high-dimensional vector (GPT-2 uses 768 dimensions; GPT-3 uses 12,288).
Similar tokens end up geometrically close in this space, which lets the model learn semantic relationships.

Two types of embeddings matter in this chapter:
- **Token embeddings** — learned lookup table mapping token ID → vector
- **Positional embeddings** — encode where in the sequence a token appears (since self-attention is order-agnostic)

> No code in section 2.1 — this is purely conceptual groundwork.

---
## 2.2 Tokenizing Text

Tokenization breaks raw text into smaller units (tokens). We build a simple regex-based tokenizer first before graduating to BPE in 2.5.

Training corpus: *The Verdict* by Edith Wharton (public domain short story, ~20 KB).

In [2]:
import os
import re
import requests
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.12.1


In [3]:
# Download training corpus (idempotent)
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)
    print("Downloaded the-verdict.txt")
else:
    print("the-verdict.txt already present")

Downloaded the-verdict.txt


In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text)}")
print(raw_text[:99])

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [5]:
# Step 1: split on whitespace (keeps delimiters via capture group)
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [6]:
# Step 2: also split on commas and periods
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [7]:
# Step 3: strip whitespace and filter empty strings
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [8]:
# Step 4: handle dashes, question marks, quotes, parentheses, etc.
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [9]:
# Apply the tokenizer to the full corpus
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"Token count: {len(preprocessed)}")
print(preprocessed[:30])

Token count: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


---
## 2.3 Converting Tokens into Token IDs

Neural networks need integers, not strings. We build a **vocabulary** — a sorted, deduplicated list of all tokens — then assign each token a unique integer index.

In [10]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocabulary size: {vocab_size}")

vocab = {token: integer for integer, token in enumerate(all_words)}

Vocabulary size: 1130


In [11]:
# First 51 entries
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [12]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Remove spaces before punctuation
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [13]:
tokenizer = SimpleTokenizerV1(vocab)

text = '"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [14]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [15]:
# Encode → decode round-trip
print(tokenizer.decode(tokenizer.encode(text)))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


---
## 2.4 Adding Special Context Tokens

The V1 tokenizer crashes on words outside the training vocabulary. We handle this with two special tokens:

- `<|unk|>` — placeholder for out-of-vocabulary words  
- `<|endoftext|>` — GPT-2's separator token, used between unrelated text segments and for padding

> GPT-2 actually uses BPE (section 2.5) and has no `<|unk|>` — it can represent *any* string with subword units. We add `<|unk|>` here as a pedagogical stepping stone.

In [16]:
# V1 raises KeyError on unknown words
try:
    tokenizer.encode("Hello, do you like tea. Is this-- a test?")
except KeyError as e:
    print(f"KeyError: {e}  ← 'Hello' not in vocab built from The Verdict")

KeyError: 'Hello'  ← 'Hello' not in vocab built from The Verdict


In [17]:
# Extend vocabulary with special tokens
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(f"Vocab size: {len(vocab)}")

# Verify special tokens are at the end
for item in list(vocab.items())[-4:]:
    print(item)

Vocab size: 1132
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [18]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Replace out-of-vocabulary tokens with <|unk|>
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [19]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
# <|endoftext|> separates two independent text sources
text = " <|endoftext|> ".join((text1, text2))
print(text)

ids = tokenizer.encode(text)
print(ids)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [20]:
print(tokenizer.decode(ids))
# 'Hello' and 'palace' → <|unk|> since they weren't in The Verdict's vocab

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


---
## 2.5 BytePair Encoding (BPE)

GPT-2 uses **Byte-Pair Encoding**: instead of a word-level vocabulary it builds a *subword* vocabulary. Unknown words are broken into known subword pieces, so the vocabulary is closed — no `<|unk|>` needed.

We use OpenAI's `tiktoken` library (Rust-backed BPE, ~5× faster than the pure-Python reference):  
- GPT-2 vocab size: **50,257** tokens  
- Encoding: `tiktoken.get_encoding("gpt2")`

> Raschka's reference: `ch02/02_bonus_bytepair-encoder/`

In [21]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))
tokenizer = tiktoken.get_encoding("gpt2")

ModuleNotFoundError: No module named 'tiktoken'

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

In [ ]:
# BPE breaks unknown words into subword pieces — no <|unk|> ever needed
unknown_word = "someunknownPlace"
ids = tokenizer.encode(unknown_word)
print(f"Token IDs : {ids}")
for tid in ids:
    print(f"  {tid:5d} → '{tokenizer.decode([tid])}'")

---
## 2.6 Data Sampling with a Sliding Window

LLMs are trained with **next-token prediction**: given tokens 1…N, predict token N+1. We create (input, target) pairs where `target = input shifted right by one`.

A **sliding window** with stride < context_length creates overlapping samples — more training signal per token, but higher risk of overfitting. Stride = context_length gives non-overlapping chunks.

**⚡ M4 Mac — DataLoader notes:**
- `num_workers=0` is the safe default on macOS. Using `num_workers > 0` forks the process, which interacts poorly with the MPS memory model and can deadlock.
- Tensors are created on CPU in the Dataset `__getitem__` (as in Raschka) and moved to MPS *in the training loop*, not here. Moving inside the DataLoader with MPS workers causes pickling errors.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(f"Encoded token count: {len(enc_text)}")

In [ ]:
# Manual sliding window — how (input, target) pairs work
enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x (input ) : {x}")
print(f"y (target) : {y}")

In [ ]:
# Same thing in human-readable form
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    target  = enc_sample[i]
    print(f"{tokenizer.decode(context)!r:40s} → {tokenizer.decode([target])!r}")

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    """Sliding-window (input, target) pairs over BPE-encoded text."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids  = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, (
            "Text too short for the requested max_length"
        )

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            # Tensors stay on CPU here; move to device in the training loop
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,   # ⚡ M4 Mac: keep at 0 — MPS + multiprocessing fork = deadlock
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset   = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

In [ ]:
# batch_size=1, context_size=4, stride=1 → highly overlapping
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
data_iter   = iter(dataloader)
first_batch = next(data_iter)
print("First batch :", first_batch)
print("Second batch:", next(data_iter))

In [ ]:
# batch_size=8, stride=max_length → non-overlapping (less overfitting risk)
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)
inputs, targets = next(iter(dataloader))
print("Inputs :\n", inputs)
print("\nTargets:\n", targets)

---
## 2.7 Creating Token Embeddings

`nn.Embedding` is a learnable lookup table: shape `[vocab_size, embedding_dim]`. Token ID `i` retrieves row `i` — equivalent to one-hot encoding followed by a linear layer, but far more memory-efficient.

**⚡ M4 Mac:** We move the embedding layer and input tensors to `device` (MPS). For Ch. 02 this is cosmetic — the real payoff is in training loops. Note that MPS requires **float32** (the default); float64 is unsupported on MPS.

In [ ]:
# Small worked example: vocab=6 tokens, 3-dimensional embeddings
torch.manual_seed(123)

small_vocab_size = 6
output_dim       = 3

# ⚡ M4 Mac: .to(device) places the embedding weight matrix on MPS
embedding_layer = torch.nn.Embedding(small_vocab_size, output_dim).to(device)
print("Embedding weight matrix (6×3):")
print(embedding_layer.weight)

In [ ]:
# Look up a single token (id=3)
token_id = torch.tensor([3], device=device)  # ⚡ M4 Mac: tensor on same device as layer
print(embedding_layer(token_id))             # returns row 3 of the weight matrix

In [ ]:
# Look up multiple token IDs at once
input_ids = torch.tensor([2, 3, 5, 1], device=device)  # ⚡ M4 Mac
print(embedding_layer(input_ids))

In [ ]:
# GPT-2 scale: 50,257-token vocab → 256-dimensional embeddings
vocab_size = 50257
output_dim = 256
max_length = 4

# ⚡ M4 Mac: entire embedding table lives on MPS
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim).to(device)

# Reload the dataloader and embed a batch
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
inputs, targets = next(iter(dataloader))

# ⚡ M4 Mac: move batch to MPS before embedding lookup
inputs = inputs.to(device)

token_embeddings = token_embedding_layer(inputs)
print(f"Token IDs shape       : {inputs.shape}")
print(f"Token embeddings shape: {token_embeddings.shape}")  # [8, 4, 256]

---
## 2.8 Encoding Word Positions

Self-attention is **permutation-invariant** — the same token gets the same embedding regardless of position. We fix this by adding **positional embeddings** to the token embeddings.

GPT-2 uses **absolute position embeddings**: a second learned lookup table of shape `[context_length, embedding_dim]`, where index `i` encodes position `i` in the sequence.

Final input to the LLM:  
```
input_embedding[pos] = token_embedding[token_id] + pos_embedding[pos]
```

> Alternative: **RoPE** (Rotary Position Embeddings) used by LLaMA/Mistral, or **ALiBi** used by MPT. These are covered in later chapters.

In [ ]:
context_length = max_length  # 4

# ⚡ M4 Mac: positional embedding table on MPS
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim).to(device)
print(f"Positional embedding table shape: {pos_embedding_layer.weight.shape}")

In [ ]:
# torch.arange(context_length) gives [0, 1, 2, 3] — the position indices
position_ids   = torch.arange(context_length, device=device)  # ⚡ M4 Mac
pos_embeddings = pos_embedding_layer(position_ids)
print(f"Positional embeddings shape: {pos_embeddings.shape}")  # [4, 256]

In [ ]:
# Combine: token embeddings [8, 4, 256] + positional embeddings [4, 256]
# Broadcasting adds pos_embeddings to every row in the batch
input_embeddings = token_embeddings + pos_embeddings
print(f"Final input embeddings shape: {input_embeddings.shape}")  # [8, 4, 256]
print(f"Tensor device: {input_embeddings.device}")

---
## Summary

The full Ch. 02 pipeline in one picture:

```
raw text  →  tokenize (regex / BPE)  →  token IDs
         →  sliding-window DataLoader
         →  token embedding lookup   ─┐
                                      ├→ add  →  input_embeddings  →  LLM
         →  positional embedding     ─┘
```

**What we built:**

| Component | Class / Function | Notes |
|-----------|-----------------|-------|
| Regex tokenizer | `SimpleTokenizerV1/V2` | Pedagogical; real models use BPE |
| BPE tokenizer | `tiktoken.get_encoding("gpt2")` | 50,257 vocab, no `<\|unk\|>` |
| Dataset | `GPTDatasetV1` | Sliding window over BPE token IDs |
| DataLoader | `create_dataloader_v1` | `num_workers=0` for MPS safety |
| Token embeddings | `nn.Embedding(50257, 256)` | Learnable lookup table |
| Positional embeddings | `nn.Embedding(ctx_len, 256)` | Absolute, GPT-2 style |

**⚡ M4 Mac changes vs. Raschka original:**
1. `device = torch.device("mps")` established at the top
2. Embedding layers and input tensors moved to MPS with `.to(device)`
3. `num_workers=0` enforced in the DataLoader (macOS fork safety)
4. Tensors created with default `float32` dtype (MPS doesn't support `float64`)

**Next:** Chapter 3 — Self-Attention and the Transformer block.